In [2]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
import os
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

c:\Users\Usuario\miniforge3\envs\alba\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def get_bert_embeddings(texts, tokenizer, bert_model, batch_size=32):
    bert_model.eval()
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            inputs = tokenizer(batch_texts, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
            outputs = bert_model(**inputs)
            cls_embeds = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(cls_embeds)
    return np.vstack(embeddings)

def cargar_configuracion_aumento(file_path):
    config = {}
    with open(file_path, 'r') as f:
        for line in f:
            if '|' in line:
                diag, valor = line.strip().split('|')
                config[diag.strip()] = float(valor.strip())
    return config

In [4]:
class VAE(nn.Module):
    def __init__(self, input_dim=768, latent_dim=32):
        super(VAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU()
        )
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, input_dim)
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        h = self.encoder(x)
        mu, logvar = self.fc_mu(h), self.fc_logvar(h)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

In [5]:
def pipeline_vae_por_modelo(csv_path, config_path, model_name, test_size=0.2, seed=42):
    print(f"\n=======================================================")
    print(f" Procesando con Modelo: {model_name}")
    print(f" Dataset: {os.path.basename(csv_path)}")
    print(f"=======================================================")
    
    # Cargar Tokenizer y Modelo BERT específico
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    bert_model = AutoModel.from_pretrained(model_name).to(device)
    
    # 1. Cargar datos
    df = pd.read_csv(csv_path, sep='|')
    
    # Identificar columnas de texto (todas menos la etiqueta 'DIAG PSQ')
    text_cols = [c for c in df.columns if c != 'DIAG PSQ']
    df['Texto_Procesar'] = df[text_cols].apply(
        lambda row: ' '.join([str(x) for x in row if pd.notna(x) and str(x).strip() != '']), axis=1
    )
    
    # 2. DIVIDIR PRIMERO EN TRAIN Y TEST REALES (Evita Fuga de Datos)
    df_train_real, df_test_real = train_test_split(
        df, test_size=test_size, stratify=df['DIAG PSQ'], random_state=seed
    )
    
    config_aumento = cargar_configuracion_aumento(config_path)
    
    train_embeddings_list = []
    test_embeddings_list = []
    
    # 3. Procesar conjunto de TEST (Únicamente Reales)
    print("\n--- Generando embeddings para TEST (Solo Reales) ---")
    test_embeds = get_bert_embeddings(df_test_real['Texto_Procesar'].tolist(), tokenizer, bert_model)
    df_test_out = pd.DataFrame(test_embeds)
    df_test_out['DIAG PSQ'] = df_test_real['DIAG PSQ'].values
    df_test_out['Origen'] = 'Real'
    test_embeddings_list.append(df_test_out)
    
    # 4. Procesar conjunto de TRAIN (Reales + VAE Sintéticos)
    print("\n--- Generando y aumentando TRAIN con VAE ---")
    for diag in df_train_real['DIAG PSQ'].unique():
        subset = df_train_real[df_train_real['DIAG PSQ'] == diag]
        porcentaje = config_aumento.get(diag, 0.0)
        
        # A) Extraer embeddings reales de Train
        embeds_reales_train = get_bert_embeddings(subset['Texto_Procesar'].tolist(), tokenizer, bert_model)
        
        df_real_train = pd.DataFrame(embeds_reales_train)
        df_real_train['DIAG PSQ'] = diag
        df_real_train['Origen'] = 'Real'
        train_embeddings_list.append(df_real_train)
        
        # B) Entrenar VAE y generar sintéticos solo si porcentaje > 0
        n_generar = int(len(subset) * porcentaje)
        if n_generar > 0:
            print(f" [{diag}] Train Reales: {len(subset)} | Generando {n_generar} sintéticos ({porcentaje*100}%)...")
            
            vae = VAE(input_dim=embeds_reales_train.shape[1], latent_dim=32).to(device)
            optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)
            X_train_tensor = torch.FloatTensor(embeds_reales_train).to(device)
            
            vae.train()
            epochs = 150 # Se aumentan épocas porque se normaliza la pérdida
            for epoch in range(epochs):
                optimizer.zero_grad()
                recon, mu, logvar = vae(X_train_tensor)
                
                # LOSS NORMALIZADO (mean)
                mse_loss = nn.functional.mse_loss(recon, X_train_tensor, reduction='mean')
                kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
                loss = mse_loss + 0.01 * kl_loss # Factor de escala para equilibrar KL y MSE
                
                loss.backward()
                optimizer.step()
            
            # Muestrear
            vae.eval()
            with torch.no_grad():
                z = torch.randn(n_generar, 32).to(device)
                sinteticos = vae.decoder(z).cpu().numpy()
            
            df_sint = pd.DataFrame(sinteticos)
            df_sint['DIAG PSQ'] = diag
            df_sint['Origen'] = 'Sintetico'
            train_embeddings_list.append(df_sint)
        else:
            print(f" [{diag}] Train Reales: {len(subset)} | Sin aumento sintético.")

    df_train_final = pd.concat(train_embeddings_list).reset_index(drop=True)
    df_test_final = pd.concat(test_embeddings_list).reset_index(drop=True)
    
    return df_train_final, df_test_final

In [ ]:
# Ejemplo de uso para BETO Cased y la variante "combinado":
modelos = {
    "beto_cased": "dccuchile/bert-base-spanish-wwm-cased",
}

path_dataset_undersampled = r'C:\Users\Usuario\Documents\Workspace\Mirage\TFG\dataset\dataset_finales\Undersampling\Dataset_procesable_undersampled.csv'
path_config = r'C:\Users\Usuario\Documents\Workspace\Mirage\code\tramo_final\config_aumento.csv'

# Generar para BETO Cased
df_train, df_test = pipeline_vae_por_modelo(
    csv_path=path_dataset_undersampled,
    config_path=path_config,
    model_name=modelos["beto_cased"]
)

# Guardar ambos datasets
df_train.to_csv('train_embeddings_combinado_beto_cased_vae.csv', index=False, sep='|')
df_test.to_csv('test_embeddings_combinado_beto_cased_vae.csv', index=False, sep='|')


 Procesando con Modelo: dccuchile/bert-base-spanish-wwm-cased
 Dataset: Dataset_procesable_undersampled.csv


Some weights of BertModel were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



--- Generando embeddings para TEST (Solo Reales) ---

--- Generando y aumentando TRAIN con VAE ---
 [F20] Train Reales: 400 | Sin aumento sintético.
 [F22] Train Reales: 216 | Generando 75 sintéticos (35.0%)...
 [F25] Train Reales: 88 | Generando 30 sintéticos (35.0%)...
 [F29] Train Reales: 119 | Generando 41 sintéticos (35.0%)...
 [F23] Train Reales: 56 | Generando 19 sintéticos (35.0%)...
 [F60.1] Train Reales: 10 | Generando 20 sintéticos (200.0%)...
 [F21] Train Reales: 6 | Generando 12 sintéticos (200.0%)...
